# Sprint 3 — MoE Full Training Pipeline

## What this notebook does and why it matters

After Sprint 1 (lung Dice ✅) and Sprint 2 (TB head AUROC ✅), the pipeline still has a critical broken path:
**every image produces ALP = 0 and Timika score = 0**, regardless of how sick the patient is.

### Root cause
The three MoE (Mixture-of-Experts) modules are implemented in code but were **never trained**:
- **Component 3** (routing gate): randomly initialized → sends nonsense weights to experts
- **Component 5** (4 expert decoders): randomly initialized → produce random logits → after thresholding, empty mask
- **Component 6** (fusion): randomly initialized → fuses garbage from garbage

An empty lesion mask means `ALP = 0` → `Timika score = 0` → the severity score is clinically useless.

### What this notebook trains

The training happens in **4 sequential phases** (order is mandatory — violating it produces empty masks):

| Phase | What runs | Time on T4 | Output |
|---|---|---|---|
| **Phase 0** (cache) | Freeze C1+C2+C4, run all images through them, save embeddings to disk | ~35 min | `embedding_cache/*.pt` |
| **Phase 1** (expert pretrain) | Train each expert decoder independently on its pathology | ~35 min | `expert_*.pt` × 4 |
| **Phase 2** (gate train) | Freeze expert weights, train routing gate only | ~20 min | `moe_best.pt` |
| **Phase 3** (boundary critic) | Train ResNet18 boundary quality scorer | ~10 min | `boundary_critic.pt` |

**Total: ~100 min on a Kaggle T4** (well within the 9-hour session limit).

### What each expert learns
| Expert | Pathology | Why it matters clinically |
|---|---|---|
| Expert 1: Consolidation | Dense opacity — lobar/segmental | Most common TB presentation |
| Expert 2: Cavity | Ring-enhancing lesion with air-space inside | Most dangerous — indicates active, infectious TB; Timika +40 pts |
| Expert 3: Fibrosis | Linear/reticular pattern, upper lobes | Indicates healing but also old disease burden |
| Expert 4: Nodule | Small focal opacity < 3 cm | Early or miliary TB |

### Datasets to attach to this notebook
| # | Kaggle dataset | What it provides |
|---|---|---|
| 1 | `iahmedhabib/montgomery` | Montgomery CXRs + lung masks |
| 2 | `iahmedhabib/shehzhenn` | Shenzhen CXRs |
| 3 | `usmanshams/tbx-11` | TBX11K CXRs + bounding box annotations |
| 4 | `iahmedhabib/medsam-vit-b` | MedSAM ViT-B checkpoint (~358 MB) |
| 5 | *(your checkpoints dataset)* | `component2_routing_head.pt`, `component4_mask_decoder.pt` |

> **NIH CXR-14 is intentionally excluded** from the MoE cache — it has no TB lesion bounding boxes so it cannot supervise the expert decoders.

---

## Verification — how you will know Sprint 3 succeeded

After training, you run **`eval_moe.ipynb`** (already exists in this repo). Sprint 3 passes when:

| Metric | Before Sprint 3 | After Sprint 3 target | What it means |
|---|---|---|---|
| `alp_mean` (all datasets) | 0.0000 | > 0.05 | Expert decoders produce non-empty lesion masks |
| `timika_auroc` (Montgomery) | 0.5000 | ≥ 0.60 | ALP score correlates with actual disease severity |
| `timika_auroc` (Shenzhen) | 0.5000 | ≥ 0.70 | Same, Shenzhen is larger so this should be higher |
| `cavity_detection_rate` | 0.0000 | > 0.10 | Expert 2 finds ring-enhancing lesions in some TB+ cases |
| Expert routing weights | All ~0.25 | Varies by dataset | Gate learned to route, not just averaging uniformly |

**The single most important check:** `alp_mean > 0`. If this is still 0, the expert decoders are not producing masks — check the troubleshooting section at the bottom of this notebook.

In [1]:
# ── Cell 1: Clone / pull repo and install dependencies ────────────────────
import os
import subprocess
import sys
from pathlib import Path

REPO_DIR = Path('/kaggle/working/repo')

if not REPO_DIR.exists():
    subprocess.run(
        ['git', 'clone', 'https://github.com/mabdullahi7780/dl-project-codebase.git', str(REPO_DIR)],
        check=True
    )
    print('Cloned fresh.')
else:
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--rebase'], check=True)
    print('Pulled latest.')

os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

# torchxrayvision required for C2 (without it TB AUROC = 0.5 — see Sprint 2 post-mortem)
subprocess.run(
    ['pip', 'install', '-r', 'requirements.txt', '-q'],
    check=True
)
subprocess.run(
    ['pip', 'install', '-q', 'torchxrayvision', 'scikit-learn', 'safetensors', 'scipy'],
    check=True
)

print('\nRepo ready:', REPO_DIR)
print('Python:', sys.executable)

Cloning into '/kaggle/working/repo'...


Cloned fresh.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 29.0/29.0 MB 62.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.8/118.8 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.4/107.4 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.8/529.8 kB 32.5 MB/s eta 0:00:00


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires ipykernel==6.17.1, but you have ipykernel 7.2.0 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.
jupyter-kernel-gateway 2.5.2 requires jupyter-client<8.0,>=5.2.0, but you have jupyter-client 8.8.0 which is incompatible.



Repo ready: /kaggle/working/repo
Python: /usr/bin/python3


In [2]:
# ── Cell 2: Verify dataset paths and prior checkpoints ────────────────────
#
# IMPORTANT: Update CHECKPOINTS_ROOT below to match your Kaggle dataset slug.
# This dataset must contain:
#   - component2_routing_head.pt   (trained in Sprint 2)
#   - component4_mask_decoder.pt   (trained in Sprint 1)
# Without these, the cache phase will use randomly-initialized C2 and C4,
# producing garbage domain_ctx and lung masks, which degrades expert supervision.
#
import torch
from pathlib import Path

KAGGLE_INPUT = Path('/kaggle/input')

# ── Dataset mounts ──────────────────────────────────────────────────────────
MONTGOMERY_ROOT = KAGGLE_INPUT / 'datasets/iahmedhabib/montgomery/montgomery'
SHENZHEN_ROOT   = KAGGLE_INPUT / 'datasets/iahmedhabib/shehzhenn/shenzhen'
TBX11K_ROOT     = KAGGLE_INPUT / 'datasets/usmanshams/tbx-11/TBX11K'
MEDSAM_CKPT     = KAGGLE_INPUT / 'datasets/iahmedhabib/medsam-vit-b/medsam_vit_b.pth'

# ── YOUR trained checkpoints — update this slug ─────────────────────────────
CHECKPOINTS_ROOT = KAGGLE_INPUT / 'datasets/mabdullahi454/tb-pipeline-checkpoints'


# Try multiple candidate paths for flexibility
def _find_file(*candidates):
    for c in candidates:
        p = Path(c)
        if p.is_file():
            return p
    return None

C2_ROUTING_HEAD = _find_file(
    CHECKPOINTS_ROOT / 'component2_routing_head.pt',
    KAGGLE_INPUT / 'component2-artifacts/component2_routing_head.pt',
)
C4_DECODER = _find_file(
    CHECKPOINTS_ROOT / 'component4_mask_decoder.pt',
    KAGGLE_INPUT / 'component4-artifacts/component4_mask_decoder.pt',
)
C1_ADAPTER = _find_file(
    CHECKPOINTS_ROOT / 'component1_adapters.safetensors',
    KAGGLE_INPUT / 'component1-artifacts/component1_adapters.safetensors',
)

print('=== Dataset Mounts ===')
required_mounts = {
    'montgomery CXR_png':   MONTGOMERY_ROOT / 'CXR_png',
    'shenzhen images':      SHENZHEN_ROOT / 'images' / 'images',
    'TBX11K imgs/':         TBX11K_ROOT / 'imgs',
    'TBX11K annotations/':  TBX11K_ROOT / 'annotations',
    'MedSAM checkpoint':    MEDSAM_CKPT,
}
all_ok = True
for label, path in required_mounts.items():
    ok = path.exists()
    print(f'  {"OK" if ok else "MISSING":6}  {label}: {path}')
    if not ok:
        all_ok = False

print('\n=== Trained Checkpoints (from Sprint 1 + 2) ===')
print(f'  {"OK" if C2_ROUTING_HEAD else "MISSING":6}  C2 routing head : {C2_ROUTING_HEAD or "not found"}')
print(f'  {"OK" if C4_DECODER     else "MISSING":6}  C4 decoder      : {C4_DECODER     or "not found"}')
print(f'  {"OK" if C1_ADAPTER     else "MISSING":6}  C1 LoRA adapter : {C1_ADAPTER     or "not found"}')

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'\nDevice: {device}')
if torch.cuda.is_available():
    gpu_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU: {torch.cuda.get_device_name(0)} ({gpu_gb:.1f} GB)')

if not all_ok:
    print('\n⚠ Some required mounts are MISSING. Fix before running training cells.')
else:
    print('\n✓ All required mounts found. Ready to proceed.')

if not C2_ROUTING_HEAD:
    print('⚠ C2 routing head not found — domain context in cache will be from random init.')
    print('  The gate will still train but will not learn domain-aware routing.')
if not C4_DECODER:
    print('⚠ C4 decoder not found — lung masks in cache will use untrained decoder.')
    print('  This degrades expert supervision quality (lesion masks will leak outside lungs).')

=== Dataset Mounts ===
  OK      montgomery CXR_png: /kaggle/input/datasets/iahmedhabib/montgomery/montgomery/CXR_png
  OK      shenzhen images: /kaggle/input/datasets/iahmedhabib/shehzhenn/shenzhen/images/images
  OK      TBX11K imgs/: /kaggle/input/datasets/usmanshams/tbx-11/TBX11K/imgs
  OK      TBX11K annotations/: /kaggle/input/datasets/usmanshams/tbx-11/TBX11K/annotations
  OK      MedSAM checkpoint: /kaggle/input/datasets/iahmedhabib/medsam-vit-b/medsam_vit_b.pth

=== Trained Checkpoints (from Sprint 1 + 2) ===
  OK      C2 routing head : /kaggle/input/datasets/mabdullahi454/tb-pipeline-checkpoints/component2_routing_head.pt
  OK      C4 decoder      : /kaggle/input/datasets/mabdullahi454/tb-pipeline-checkpoints/component4_mask_decoder.pt
  OK      C1 LoRA adapter : /kaggle/input/datasets/mabdullahi454/tb-pipeline-checkpoints/component1_adapters.safetensors

Device: cuda
GPU: Tesla T4 (15.6 GB)

✓ All required mounts found. Ready to proceed.


In [3]:
# ── Cell 3: Patch configs/moe.yaml with trained checkpoint paths ──────────
#
# Why: kaggle_moe_train.py reads configs/moe.yaml and generates a runtime config.
# The base moe.yaml has routing_head_path: null for C2. We must set it before
# training so the cache phase produces meaningful domain_ctx vectors.
# domain_ctx is passed to the routing gate — garbage domain_ctx = the gate
# cannot learn which dataset it is looking at.
#
import yaml
from pathlib import Path

MOE_CONFIG_PATH = REPO_DIR / 'configs' / 'moe.yaml'
with MOE_CONFIG_PATH.open() as f:
    moe_cfg = yaml.safe_load(f)

# Set Sprint 1 and Sprint 2 checkpoint paths
if C2_ROUTING_HEAD:
    moe_cfg.setdefault('component2', {})['routing_head_path'] = str(C2_ROUTING_HEAD)
    print(f'Set C2 routing_head_path: {C2_ROUTING_HEAD}')
else:
    moe_cfg.setdefault('component2', {})['routing_head_path'] = None
    print('C2 routing_head_path: null (random init — domain routing will be weaker)')

if C4_DECODER:
    moe_cfg.setdefault('component4', {})['decoder_checkpoint_path'] = str(C4_DECODER)
    print(f'Set C4 decoder_checkpoint_path: {C4_DECODER}')
else:
    moe_cfg.setdefault('component4', {})['decoder_checkpoint_path'] = None
    print('C4 decoder: null (random init — lung masks will be coarser)')

if C1_ADAPTER:
    moe_cfg.setdefault('component1', {})['adapter_path'] = str(C1_ADAPTER)
    print(f'Set C1 adapter_path: {C1_ADAPTER}')

# Also disable the local resume_from_moe_best (won't exist on Kaggle)
moe_cfg.setdefault('moe_training', {}).setdefault('joint', {})['resume_from_moe_best'] = None

# Write back to repo
with MOE_CONFIG_PATH.open('w') as f:
    yaml.safe_dump(moe_cfg, f, sort_keys=False)

print('\n✓ configs/moe.yaml patched with Kaggle paths.')
print('  Phase 0 cache will now use trained C2/C4 weights for higher-quality supervision.')

Set C2 routing_head_path: /kaggle/input/datasets/mabdullahi454/tb-pipeline-checkpoints/component2_routing_head.pt
Set C4 decoder_checkpoint_path: /kaggle/input/datasets/mabdullahi454/tb-pipeline-checkpoints/component4_mask_decoder.pt
Set C1 adapter_path: /kaggle/input/datasets/mabdullahi454/tb-pipeline-checkpoints/component1_adapters.safetensors

✓ configs/moe.yaml patched with Kaggle paths.
  Phase 0 cache will now use trained C2/C4 weights for higher-quality supervision.


## Smoke Test — Run This First

Before committing to a 100-minute training run, run a **smoke test** with:
- 8 images per dataset (not 500)
- 1 epoch per phase (not 10/15/5)

**What you're verifying:**
1. All 4 phases complete without crashing
2. The cache `.pt` files are created and have the right keys
3. Expert checkpoints are saved after Phase 1
4. `moe_best.pt` is created after Phase 2
5. `boundary_critic.pt` is created after Phase 3

**Expected output:** ~5-8 minutes. If it crashes, fix the error before running the full training cell.

**The smoke test does NOT produce useful weights** — it just validates that all pipes are connected.

In [4]:
# ── Cell 4: Smoke test — 8 images, 1 epoch each phase (~5-8 min) ─────────
#
# Run this BEFORE the full training cell.
# If this crashes, do not run Cell 5 — fix the error first.
#
import subprocess
import sys
from pathlib import Path

SMOKE_CACHE = Path('/kaggle/working/moe_smoke_cache')
SMOKE_OUT   = Path('/kaggle/working/moe_smoke_runs')

env_patch = {
    'MONTGOMERY_ROOT': str(MONTGOMERY_ROOT),
    'SHENZHEN_ROOT':   str(SHENZHEN_ROOT),
    'TBX11K_ROOT':     str(TBX11K_ROOT),
    'MEDSAM_VIT_B_CKPT': str(MEDSAM_CKPT),
    **__import__('os').environ,
}

cmd = [
    sys.executable, 'scripts/kaggle_moe_train.py',
    '--mode', 'smoke',
    '--phase', 'all',
    '--cache-dir', str(SMOKE_CACHE),
]
print('Running smoke test...')
print('Command:', ' '.join(cmd))
print('-' * 60)

result = subprocess.run(cmd, env=env_patch, capture_output=False, text=True)

print('-' * 60)
if result.returncode == 0:
    print('\n✓ Smoke test PASSED. All 4 phases completed successfully.')
    # Verify expected artifacts
    for fname in ['expert_consolidation_best.pt', 'expert_cavity_best.pt',
                  'expert_fibrosis_best.pt', 'expert_nodule_best.pt',
                  'moe_best.pt', 'boundary_critic.pt']:
        # kaggle_moe_train.py saves to moe_runs, not smoke dir
        found = any(Path('/kaggle/working').rglob(fname))
        print(f'  {"✓" if found else "✗ MISSING":5}  {fname}')
    print('\nReady to run full training (Cell 5).')
else:
    print(f'\n✗ Smoke test FAILED (exit code {result.returncode}).')
    print('Fix the error above before running the full training cell.')

Running smoke test...
Command: /usr/bin/python3 scripts/kaggle_moe_train.py --mode smoke --phase all --cache-dir /kaggle/working/moe_smoke_cache
------------------------------------------------------------
Mode: smoke  preset={'limit_per_domain': 8, 'pretrain_epochs': 1, 'joint_epochs': 1, 'critic_epochs': 1, 'batch_size': 2}  phase=all
Kaggle mounts:
  montgomery    : /kaggle/input/datasets/iahmedhabib/montgomery/montgomery
  shenzhen      : /kaggle/input/datasets/iahmedhabib/shehzhenn/shenzhen
  tbx11k        : /kaggle/input/datasets/usmanshams/tbx-11/TBX11K
  nih_cxr14     : MISSING (optional) at /kaggle/input/datasets/organizations/nih-chest-xrays/data
  medsam_vit_b  : /kaggle/input/datasets/iahmedhabib/medsam-vit-b/medsam_vit_b.pth
  c1_adapter    : /kaggle/input/datasets/mabdullahi454/tb-pipeline-checkpoints/component1_adapters.safetensors
  c4_decoder    : /kaggle/input/datasets/mabdullahi454/tb-pipeline-checkpoints/component4_mask_decoder.pt
=== Phase 0: cache embeddings → /ka

## Full Training Run

This runs all 4 phases with production settings:

| Phase | Images | Epochs | Est. time |
|---|---|---|---|
| Phase 0: Cache | 500/domain (~1100 total) | — | ~35 min |
| Phase 1: Expert pretrain | Same cache | 10 per expert × 4 experts | ~35 min |
| Phase 2: Gate train (gate_only=true) | Same cache | 15 | ~20 min |
| Phase 3: Boundary critic | Same cache | 5 | ~10 min |
| **Total** | | | **~100 min** |

### What Phase 0 (cache) actually does
For each image:
1. Runs C1 (MedSAM) → `img_emb [256, 64, 64]`
2. Runs C2 (TXV DenseNet + routing head) → `domain_ctx [256]`
3. Runs C4 (MedSAM lung decoder) → `lung_mask [1, 256, 256]`
4. Runs `baseline_lesion_proposer` (Grad-CAM) → `lesion_mask [1, 256, 256]`
5. For TBX11K: converts bounding boxes → `gt_mask [1, 256, 256]` (actual ground truth)
6. For Montgomery/Shenzhen: uses the CAM-based pseudo-mask as weak supervision
7. Saves all of this to a `.pt` file

This pre-computation is why subsequent phases are fast — they only train small CNNs on cached features.

### What Phase 1 (expert pretrain) does
Each expert decoder (4 × ~0.5M params) is trained **independently** on its supervision signal:
- **Expert 1 (consolidation)**: trained on TXV CAM maps weighted by `consolidation + infiltration + pneumonia + lung_opacity` logits
- **Expert 2 (cavity)**: trained on TXV CAM maps weighted by `lung_lesion` logit (closest proxy to ring-enhancing lesions)
- **Expert 3 (fibrosis)**: trained on TXV CAM maps weighted by `fibrosis + pleural_thickening` logits
- **Expert 4 (nodule)**: trained on TXV CAM maps weighted by `nodule + mass` logits

TBX11K images with bounding boxes get higher supervision weight (`0.6`) than pseudo-masked images (`0.45`).

### What Phase 2 (gate training) does
The gate is trained to learn **which expert is most relevant for a given image**. With `gate_only=true`:
- Expert decoders are **frozen** (their weights do not change)
- Only the routing gate (~50K params) updates
- Loss = mask quality (BCE+Dice on fused mask vs GT) + load-balance penalty
- Load-balance prevents the gate from always routing to Expert 1 and ignoring the rest

### What Phase 3 (boundary critic) does
A ResNet18 is trained to predict boundary quality score ∈ [0,1] from (image, mask) pairs.
This replaces the heuristic boundary scorer in Component 7. Higher score = better boundary alignment with image gradients.

In [5]:
# ── Cell 5: Full MoE training run (~100 min) ──────────────────────────────
#
# Run smoke test (Cell 4) first.
# Output artifacts are saved to /kaggle/working/moe_runs/ and mirrored
# to /kaggle/working/moe_artifacts/ for download from the Output tab.
#
import subprocess
import sys
import os
import time
from pathlib import Path

# Set to 'short' for a 20-min validation run (100 images, 3 epochs),
# or 'full' for the production 100-min run (500 images, 10/15/5 epochs).
TRAINING_MODE = 'full'

FULL_CACHE = Path('/kaggle/working/moe_full_cache')

env_patch = {
    **os.environ,
    'MONTGOMERY_ROOT':    str(MONTGOMERY_ROOT),
    'SHENZHEN_ROOT':      str(SHENZHEN_ROOT),
    'TBX11K_ROOT':        str(TBX11K_ROOT),
    'MEDSAM_VIT_B_CKPT':  str(MEDSAM_CKPT),
    'PYTORCH_CUDA_ALLOC_CONF': 'expandable_segments:True',
}

cmd = [
    sys.executable, 'scripts/kaggle_moe_train.py',
    '--mode', TRAINING_MODE,
    '--phase', 'all',
    '--cache-dir', str(FULL_CACHE),
]

print(f'Starting MoE training in mode: {TRAINING_MODE}')
print('Command:', ' '.join(cmd))
print('=' * 70)

t0 = time.time()
result = subprocess.run(cmd, env=env_patch, capture_output=False, text=True)
elapsed = time.time() - t0

print('=' * 70)
print(f'Finished in {elapsed/60:.1f} min  |  Exit code: {result.returncode}')

if result.returncode != 0:
    print('\n✗ Training FAILED. Check the error above.')
    print('Common causes:')
    print('  - OOM: set TRAINING_MODE="short" or reduce batch_size in configs/moe.yaml')
    print('  - Missing cache: check that Phase 0 ran (look for /kaggle/working/moe_full_cache/*.pt)')
    print('  - Missing expert .pt after Phase 1: check /kaggle/working/moe_runs/')
else:
    print('\n✓ Training completed successfully.')

Starting MoE training in mode: full
Command: /usr/bin/python3 scripts/kaggle_moe_train.py --mode full --phase all --cache-dir /kaggle/working/moe_full_cache
Mode: full  preset={'limit_per_domain': 1000, 'pretrain_epochs': 10, 'joint_epochs': 15, 'critic_epochs': 5, 'batch_size': 4}  phase=all
Kaggle mounts:
  montgomery    : /kaggle/input/datasets/iahmedhabib/montgomery/montgomery
  shenzhen      : /kaggle/input/datasets/iahmedhabib/shehzhenn/shenzhen
  tbx11k        : /kaggle/input/datasets/usmanshams/tbx-11/TBX11K
  nih_cxr14     : MISSING (optional) at /kaggle/input/datasets/organizations/nih-chest-xrays/data
  medsam_vit_b  : /kaggle/input/datasets/iahmedhabib/medsam-vit-b/medsam_vit_b.pth
  c1_adapter    : /kaggle/input/datasets/mabdullahi454/tb-pipeline-checkpoints/component1_adapters.safetensors
  c4_decoder    : /kaggle/input/datasets/mabdullahi454/tb-pipeline-checkpoints/component4_mask_decoder.pt
=== Phase 0: cache embeddings → /kaggle/working/moe_full_cache ===
Component 2: 

In [6]:
# ── Cell 6: Verify artifacts and check ALP is non-zero ────────────────────
#
# This cell does 3 things:
# 1. Lists all saved checkpoint files with sizes
# 2. Loads moe_best.pt and inspects the gate weights
# 3. Runs one image through the full MoE pipeline to verify ALP > 0
#
import torch
import json
from pathlib import Path

MOE_RUNS = Path('/kaggle/working/moe_runs')
ARTIFACTS = Path('/kaggle/working/moe_artifacts')

print('=== Checkpoint Artifacts ===')
expected_files = [
    'expert_consolidation_best.pt',
    'expert_cavity_best.pt',
    'expert_fibrosis_best.pt',
    'expert_nodule_best.pt',
    'moe_best.pt',
    'boundary_critic.pt',
]
moe_best_path = None
for fname in expected_files:
    for search_dir in [MOE_RUNS, ARTIFACTS, Path('/kaggle/working')]:
        p = search_dir / fname
        if p.is_file():
            size_mb = p.stat().st_size / 1e6
            print(f'  ✓  {fname:<40}  {size_mb:7.2f} MB  ({p})')
            if fname == 'moe_best.pt':
                moe_best_path = p
            break
    else:
        print(f'  ✗  {fname:<40}  MISSING')

# ── Inspect moe_best.pt ──────────────────────────────────────────────────
if moe_best_path:
    print('\n=== moe_best.pt contents ===')
    ckpt = torch.load(moe_best_path, map_location='cpu', weights_only=False)
    for key in ckpt.keys():
        if isinstance(ckpt[key], dict):
            n_params = sum(v.numel() for v in ckpt[key].values() if isinstance(v, torch.Tensor))
            print(f'  key={key:<20}  n_params={n_params:,}')
        else:
            print(f'  key={key:<20}  type={type(ckpt[key])}')

    # Quick gate weight sanity check: if all experts have equal weight on every image,
    # the gate learned nothing (it's still uniform routing).
    if 'routing_gate' in ckpt:
        gate_sd = ckpt['routing_gate']
        final_layer = [v for k, v in gate_sd.items() if 'weight' in k][-1]
        weight_std = final_layer.std().item()
        print(f'\n  Routing gate final layer weight std: {weight_std:.4f}')
        if weight_std < 0.01:
            print('  ⚠ Gate weights are nearly uniform — gate may not have learned routing.')
            print('  This can happen if load_balance_weight is too high or epochs are too few.')
        else:
            print('  ✓ Gate weights show variation — gate learned non-uniform routing.')
else:
    print('\n✗ moe_best.pt not found — Phase 2 may not have completed.')

print('\nDownload files from: /kaggle/working/moe_artifacts/')

=== Checkpoint Artifacts ===
  ✓  expert_consolidation_best.pt                 0.64 MB  (/kaggle/working/moe_runs/expert_consolidation_best.pt)
  ✓  expert_cavity_best.pt                        0.64 MB  (/kaggle/working/moe_runs/expert_cavity_best.pt)
  ✓  expert_fibrosis_best.pt                      0.64 MB  (/kaggle/working/moe_runs/expert_fibrosis_best.pt)
  ✓  expert_nodule_best.pt                        0.64 MB  (/kaggle/working/moe_runs/expert_nodule_best.pt)
  ✓  moe_best.pt                                  2.97 MB  (/kaggle/working/moe_runs/moe_best.pt)
  ✓  boundary_critic.pt                          44.79 MB  (/kaggle/working/moe_runs/boundary_critic.pt)

=== moe_best.pt contents ===
  key=routing_gate          n_params=99,204
  key=expert_bank           n_params=634,512
  key=fusion                n_params=0
  key=epoch                 type=<class 'int'>

  Routing gate final layer weight std: 0.0943
  ✓ Gate weights show variation — gate learned non-uniform routing.

Downlo

In [7]:
# ── Cell 7: Single-image MoE inference sanity check ───────────────────────
#
# Runs one Montgomery TB-positive image through the full MoE path.
# KEY CHECK: ALP must be > 0.0 after Sprint 3.
# If ALP is still 0, the expert masks are still empty — see troubleshooting.
#
import torch
import glob
import sys
from pathlib import Path

# Find a TB-positive Montgomery image (filename ends _1 = TB)
mont_tb_images = sorted(MONTGOMERY_ROOT.glob('CXR_png/*_1.png'))
if not mont_tb_images:
    print('No Montgomery TB images found — check path.')
else:
    test_image = mont_tb_images[0]
    print(f'Test image (TB+): {test_image.name}')

    # Write a minimal infer config pointing at our MoE checkpoint
    import yaml
    moe_best = moe_best_path or next(
        (p for d in [Path('/kaggle/working/moe_runs'), Path('/kaggle/working/moe_artifacts')]
         for p in d.glob('moe_best.pt')), None
    )
    if not moe_best:
        print('\n✗ moe_best.pt not found — cannot run inference. Complete training first.')
    else:
        from src.app.infer import run_single_image_inference
        import yaml

        # Build minimal runtime config
        with (REPO_DIR / 'configs/moe.yaml').open() as f:
            infer_cfg = yaml.safe_load(f)
        infer_cfg['moe']['checkpoint_path'] = str(moe_best)
        infer_cfg['component1']['checkpoint_path'] = str(MEDSAM_CKPT)
        infer_cfg['component4']['checkpoint_path'] = str(MEDSAM_CKPT)
        if C4_DECODER:
            infer_cfg['component4']['decoder_checkpoint_path'] = str(C4_DECODER)
        if C2_ROUTING_HEAD:
            infer_cfg['component2']['routing_head_path'] = str(C2_ROUTING_HEAD)
        infer_cfg.setdefault('runtime', {})['device'] = 'cuda' if torch.cuda.is_available() else 'cpu'

        infer_cfg_path = Path('/kaggle/working/infer_sanity.yaml')
        with infer_cfg_path.open('w') as f:
            yaml.safe_dump(infer_cfg, f)

        out_dir = Path('/kaggle/working/sanity_output')
        out_dir.mkdir(exist_ok=True)

        print('Running MoE inference on one image...')
        try:
            bundle = run_single_image_inference(
                image_path=str(test_image),
                dataset_id='montgomery',
                config_path=str(infer_cfg_path),
                outdir=str(out_dir),
            )
            print('\n=== MoE Inference Result ===')
            print(f'  ALP:          {bundle.alp}')
            print(f'  Cavity flag:  {bundle.cavity_flag}')
            print(f'  Timika score: {bundle.timika_score}')
            print(f'  Severity:     {bundle.severity}')

            if bundle.alp is not None and bundle.alp > 0:
                print('\n✓ ALP > 0 — Expert decoders are producing non-empty lesion masks!')
                print('  Sprint 3 core objective ACHIEVED.')
            else:
                print('\n✗ ALP = 0 — Expert masks are still empty!')
                print('  See troubleshooting in the verification cell below.')
        except Exception as e:
            print(f'\nInference error: {e}')
            print('This may be a config path issue — check the error and retry.')

Test image (TB+): MCUCXR_0104_1.png
Running MoE inference on one image...

Inference error: run_single_image_inference() got an unexpected keyword argument 'dataset_id'
This may be a config path issue — check the error and retry.


## How to Verify Sprint 3 Results — Complete Guide

### Step 1: Download the checkpoint files

After the training cell completes, go to the Kaggle **Output** tab and download:
```
/kaggle/working/moe_artifacts/moe_best.pt          ← Main MoE checkpoint
/kaggle/working/moe_artifacts/boundary_critic.pt    ← Optional: boundary critic
/kaggle/working/moe_artifacts/expert_*.pt           ← Individual expert files
```

### Step 2: Upload to your checkpoints Kaggle dataset

Add these files to your `tb-pipeline-checkpoints` dataset (or create a new version).
The eval_moe.ipynb Cell 3 already handles assembling individual `expert_*.pt` files into a
temporary `moe_assembled.pt` file automatically.

### Step 3: Run eval_moe.ipynb

`eval_moe.ipynb` is already in this repo. It:
1. Attaches your checkpoints dataset (with the new MoE files)
2. Runs the full MoE pipeline on Montgomery + Shenzhen + TBX11K + NIH
3. Outputs `moe_system.csv`, `moe_components.csv`, `moe_per_image.csv`
4. Generates comparison charts vs the baseline

---

### What good numbers look like

| Metric | Bad (Sprint 3 failed) | Good (Sprint 3 passed) | SOTA target |
|---|---|---|---|
| `alp_mean` (Montgomery) | 0.0000 | 0.05 – 0.25 | 0.10 – 0.30 |
| `alp_mean` (Shenzhen) | 0.0000 | 0.05 – 0.30 | 0.10 – 0.35 |
| `timika_auroc` (Montgomery) | 0.5000 | 0.55 – 0.70 | 0.70 – 0.78 |
| `timika_auroc` (Shenzhen) | 0.5000 | 0.65 – 0.75 | 0.72 – 0.80 |
| `cavity_detection_rate` | 0.0000 | 0.05 – 0.20 | 0.20+ |
| `moe_lesion_dice` (TBX11K) | N/A (empty) | 0.20 – 0.40 | 0.40 – 0.65 |
| Expert routing weights | All ≈ 0.25 | Varies by domain | Dataset-specific routing |

**The expert routing weight distribution (Cell 8 of eval_moe.ipynb)** is a key diagnostic:
- If all 4 weights are ~0.25 for all datasets → gate is still routing uniformly (didn't learn)
- If Montgomery routes more to Expert 3 (fibrosis) and TBX11K more to Expert 1 (consolidation) → gate learned pathology-aware routing

---

### Troubleshooting — if ALP is still 0 after training

Work through these in order:

**Problem 1: Expert files not loaded into moe_best.pt**
```
Symptom: moe_best.pt exists but expert_bank has only random weights
Fix: Check Cell 6 output — look for "Loaded pretrained expert: X" in Phase 2 log.
     If missing, expert_*.pt files were saved to wrong directory.
     Check: ls /kaggle/working/moe_runs/expert_*.pt
```

**Problem 2: Cache is empty or has wrong keys**
```
Symptom: Phase 1 crashes with KeyError on expert_masks
Fix: The cache_moe_embeddings.py failed silently. Check:
     import torch
     pt = torch.load('/kaggle/working/moe_full_cache/montgomery__XXXX.pt', weights_only=False)
     print(pt.keys())  # should include: image_emb, domain_ctx, lung_mask, lesion_mask, expert_masks
     If expert_masks is missing, rerun Phase 0 only:
     python scripts/kaggle_moe_train.py --mode full --phase cache
```

**Problem 3: Inference config not finding moe_best.pt**
```
Symptom: eval_moe.ipynb says 'MoE checkpoint: NOT FOUND'
Fix: Check Cell 3 of eval_moe.ipynb — update CHECKPOINTS_ROOT to your dataset slug.
     Verify moe_best.pt (or individual expert_*.pt) are in your Kaggle dataset.
```

**Problem 4: Expert masks are non-empty but ALP is still 0**
```
Symptom: moe_lesion_dice > 0 in eval but alp_mean = 0
Fix: The lung mask is empty (C4 decoder issue) — ALP = lesion/lung, if lung=0 → ALP=0.
     Check: c4_lung_dice in eval_moe.ipynb moe_components.csv
     If lung Dice is 0, re-upload component4_mask_decoder.pt (Sprint 1 checkpoint).
```

**Problem 5: OOM crash during cache phase**
```
Symptom: CUDA out of memory during Phase 0
Fix: In configs/moe.yaml, set batch_size in moe_cache or reduce in kaggle_moe_train.py.
     Or: use TRAINING_MODE = 'short' (100 images, smaller GPU footprint).
```

---

### After Sprint 3: What to improve next (towards SOTA)

Once Timika AUROC is non-trivial (> 0.55), these improvements push towards literature SOTA:

1. **Per-dataset threshold calibration** — the mask threshold (0.5) is global. Sweeping threshold per dataset can recover 0.03–0.05 Dice.

2. **Full MoE joint training** (`gate_only: false`) — currently only the gate trains in Phase 2. Setting `gate_only: false` in `configs/moe.yaml` allows all 3 modules to co-train. Increases epochs to 20. Risk: expert mode-collapse (all routing → Expert 1). Monitor load-balance loss.

3. **Montgomery oversampling** — only ~110 images vs 7000+ TBX11K. Set `domain_sampling_weights.montgomery: 5.0` in the expert pretraining config to compensate. This should raise Montgomery-specific Timika AUROC.

4. **Progressive expert fine-tuning** — after gate trains, unfreeze Expert 2 (cavity) only and retrain for 5 epochs with TBX11K cavity-annotated samples. Cavity detection rate is the hardest metric to move.

5. **TTA (test-time augmentation)** — average predictions across horizontal flip + ±5° rotation. Improves Dice 0.01–0.02 at zero training cost.

In [8]:
# ── Cell 8: Print training log summary ───────────────────────────────────
#
# The training script writes JSONL logs for each phase.
# This cell reads them and prints a summary table.
#
import json
from pathlib import Path

MOE_RUNS = Path('/kaggle/working/moe_runs')

print('=== Phase 1: Expert Pretraining Loss Summary ===')
for expert_name in ['consolidation', 'cavity', 'fibrosis', 'nodule']:
    log_file = MOE_RUNS / f'expert_{expert_name}_train.jsonl'
    if log_file.exists():
        lines = [json.loads(l) for l in log_file.read_text().splitlines() if l.strip()]
        if lines:
            first = lines[0]
            last  = lines[-1]
            print(f'  {expert_name:<15}  epoch 1 loss={first.get("train_loss", "?"):.4f}  '
                  f'→  epoch {last.get("epoch", "?")} loss={last.get("train_loss", "?"):.4f}  '
                  f'val_dice={last.get("val_dice", float("nan")):.4f}')
    else:
        print(f'  {expert_name:<15}  log not found at {log_file}')

print('\n=== Phase 2: Joint MoE Training Loss Summary ===')
joint_log = MOE_RUNS / 'moe_joint_train.jsonl'
if joint_log.exists():
    lines = [json.loads(l) for l in joint_log.read_text().splitlines() if l.strip()]
    if lines:
        print(f'  Epochs run: {len(lines)}')
        print(f'  Epoch 1: loss={lines[0].get("train_loss", "?"):.4f}')
        print(f'  Epoch N: loss={lines[-1].get("train_loss", "?"):.4f}')
        if 'val_dice' in lines[-1]:
            print(f'  Final val_dice: {lines[-1]["val_dice"]:.4f}')
        if 'load_balance_loss' in lines[-1]:
            lb = lines[-1]['load_balance_loss']
            print(f'  Final load_balance_loss: {lb:.4f}  '
                  f'({"OK" if lb < 0.05 else "HIGH — gate may be routing to one expert only"})')
else:
    print(f'  log not found at {joint_log}')

print('\n=== Phase 3: Boundary Critic ===')
critic_log = MOE_RUNS / 'boundary_critic_train.jsonl'
if critic_log.exists():
    lines = [json.loads(l) for l in critic_log.read_text().splitlines() if l.strip()]
    if lines:
        print(f'  Epochs run: {len(lines)}')
        print(f'  Final val_acc: {lines[-1].get("val_acc", "?")}')
else:
    print(f'  log not found at {critic_log}')

print('\n=== Quick interpretation guide ===')
print('  Expert loss should DROP from epoch 1 to N (learning to generate masks).')
print('  If expert loss is flat (~0.7) — expert is outputting all-zeros (mode collapse).')
print('  Joint val_dice > 0.10 means fused mask is non-trivially overlapping GT.')
print('  load_balance_loss < 0.05 means gate is not degenerate (not all routing to 1 expert).')

=== Phase 1: Expert Pretraining Loss Summary ===
  consolidation    log not found at /kaggle/working/moe_runs/expert_consolidation_train.jsonl
  cavity           log not found at /kaggle/working/moe_runs/expert_cavity_train.jsonl
  fibrosis         log not found at /kaggle/working/moe_runs/expert_fibrosis_train.jsonl
  nodule           log not found at /kaggle/working/moe_runs/expert_nodule_train.jsonl

=== Phase 2: Joint MoE Training Loss Summary ===
  log not found at /kaggle/working/moe_runs/moe_joint_train.jsonl

=== Phase 3: Boundary Critic ===
  log not found at /kaggle/working/moe_runs/boundary_critic_train.jsonl

=== Quick interpretation guide ===
  Expert loss should DROP from epoch 1 to N (learning to generate masks).
  If expert loss is flat (~0.7) — expert is outputting all-zeros (mode collapse).
  Joint val_dice > 0.10 means fused mask is non-trivially overlapping GT.
  load_balance_loss < 0.05 means gate is not degenerate (not all routing to 1 expert).
